<h1>Job Recommender<h1>

In [1]:
import pandas as pd
import re
import ast
import joblib
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# ✅ Suppress convergence warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# 🧠 Load new dataset
data = pd.read_csv('jobs_data.csv')
data.columns = data.columns.str.strip().str.lower()

# 🧹 Drop missing/empty rows
data.dropna(subset=['resume', 'category'], inplace=True)
data = data[data['resume'].str.strip() != '']

# 🧼 Clean resume text
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove symbols
    text = re.sub(r'\s+', ' ', text)  # Extra spaces
    text = text.lower().strip()
    return ' '.join([w for w in text.split() if w not in ENGLISH_STOP_WORDS])

data['resume'] = data['resume'].apply(clean_text)

# 🔧 Convert string to list using ast.literal_eval
data['category'] = data['category'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

# 🏷️ Binarize job categories
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data['category'])

# ✨ TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(data['resume'])

# 🔀 Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🤖 Train model with better settings
model = OneVsRestClassifier(LogisticRegression(max_iter=2000, solver='saga', class_weight='balanced'))
model.fit(X_train, y_train)

# 📊 Evaluate model using probability threshold
y_probs = model.predict_proba(X_test)
threshold = 0.3
y_pred = (y_probs >= threshold).astype(int)

print("\n📊 Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=mlb.classes_, zero_division=0))

# 💾 Save files
joblib.dump(model, 'job_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
joblib.dump(mlb, 'label_binarizer.pkl')

print("\n✅ Multi-label classification model trained and saved successfully with probability-based evaluation.")



📊 Classification Report:

                           precision    recall  f1-score   support

            AI Researcher       0.13      1.00      0.23       506
               BI Analyst       0.13      1.00      0.23       516
        Backend Developer       0.13      0.98      0.23       524
     Blockchain Developer       0.12      0.99      0.22       490
          Cloud Architect       0.12      1.00      0.21       465
          DevOps Engineer       0.13      1.00      0.23       521
Embedded Systems Engineer       0.13      1.00      0.23       511
     Full Stack Developer       0.12      0.99      0.22       482
           Game Developer       0.12      1.00      0.22       498
              ML Engineer       0.12      0.99      0.22       489
         Mobile Developer       0.12      1.00      0.22       491
          Product Manager       0.14      1.00      0.24       540
              QA Engineer       0.12      1.00      0.22       487
         Security Analyst       0.